In [5]:
from tdmpc2lora_tmp.config import Config
from tdmpc2lora_tmp.train_env import make_env
from tdmpc2lora_tmp.model import TDMPC2 # model.pyにリネーム済みと仮定
from tdmpc2lora_tmp.trainer import OnlineTrainer

In [ ]:
DO_TEST_0 = False
if DO_TEST_0:
    # 実験設定
    cfg = Config()
    cfg.task = "SimpleReacher"
    cfg.task_id = 0
    cfg.lora_rank = 4 # LoRA有効化

    # 環境・エージェント作成
    env = make_env(cfg)
    agent = TDMPC2(cfg)
    
    # 学習開始
    trainer = OnlineTrainer(cfg, env, agent)
    trainer.train()

----
----
----

In [ ]:
import torch
from tdmpc2lora_tmp.config import Config
from tdmpc2lora_tmp.train_env import make_env
from tdmpc2lora_tmp.model import TDMPC2
from tdmpc2lora_tmp.trainer import OnlineTrainer

def train_task(task_id, lora_rank, prev_model_path=None):
    print(f"\n=== Training Task {task_id} (LoRA Rank: {lora_rank}) ===")
    
    # 1. 設定
    cfg = Config()
    cfg.task = "SimpleReacher"
    cfg.task_id = task_id
    cfg.lora_rank = lora_rank
    cfg.steps = 5000 # 確認ようなので短く
    
    # 実験名に工夫を入れる（Task0 -> Task1 の流れがわかるように）
    base_name = f"Reacher_Task{task_id}_rank{lora_rank}"
    if prev_model_path:
        base_name += "_continued"
    # Configのrun_nameをハックして上書き（簡易的な方法）
    cfg.__class__.run_name = property(lambda self: base_name)

    # 2. 環境・エージェント作成
    env = make_env(cfg)
    agent = TDMPC2(cfg)
    # print("--- Parameter Names Check ---")
    # for name, param in agent.model.named_parameters():
    #     # LoRAパラメータかどうかを判定して表示
    #     is_lora = "lora" in name
    #     print(f"{'[LoRA]' if is_lora else '[Base]'} {name}")

    # 3. 過去のモデルのロード（継続学習の場合）
    if prev_model_path:
        print(f"Loading model from: {prev_model_path}")
        # CPUでロード
        ckpt = torch.load(prev_model_path, map_location="cpu", weights_only=False)
        
        # --- 重要: 重みのロード処理 ---
        # 形状が違う（LoRAが増える等）とエラーになるので、strict=Falseでロードし
        # 共通部分（Core）だけを復元する挙動を確認する
        keys = agent.model.load_state_dict(ckpt["model"], strict=False)
        print(f"Loaded keys: {len(keys.missing_keys)} missing, {len(keys.unexpected_keys)} unexpected")
        
        # 共通部分（LoRA以外）を固定するかどうかのロジックもここでテスト
        for name, param in agent.model.named_parameters():
            if "lora" not in name:
                param.requires_grad = False
                print(f"[Base] Param: {name}, requires_grad={param.requires_grad}")
            else:
                print(f"[LoRA] Param: {name}, requires_grad={param.requires_grad}")
    
    # 4. 学習開始
    trainer = OnlineTrainer(cfg, env, agent)
    trainer.train()
    
    # 保存されたベストモデルのパスを返す
    return cfg.get_model_dir() / "best.pth"

# --- 実験フロー ---
if __name__ == "__main__":
    # Phase 1: Task 0 を学習 (LoRAなし または あり)
    model_path_task0 = train_task(task_id=0, lora_rank=4, prev_model_path=None)
    
    # # Phase 2: Task 1 を学習 (Task 0 のモデルを引き継ぐ)
    # # ここでエラーが出なければ、継続学習のパイプラインは完成です！
    # train_task(task_id=1, lora_rank=4, prev_model_path=model_path_task0)


=== Training Task 0 (LoRA Rank: 4) ===
[Env] Created SimpleReacherEnv for Task 0 (Target: [1. 1.])
--- Parameter Names Check ---
[LoRA] _encoder.0.lora_A
[LoRA] _encoder.0.lora_B
[Base] _encoder.0.linear.weight
[Base] _encoder.0.linear.bias
[Base] _encoder.0.ln.weight
[Base] _encoder.0.ln.bias
[Base] _encoder.2.weight
[Base] _encoder.2.bias
[LoRA] _dynamics.0.lora_A
[LoRA] _dynamics.0.lora_B
[Base] _dynamics.0.linear.weight
[Base] _dynamics.0.linear.bias
[Base] _dynamics.0.ln.weight
[Base] _dynamics.0.ln.bias
[LoRA] _dynamics.2.lora_A
[LoRA] _dynamics.2.lora_B
[Base] _dynamics.2.linear.weight
[Base] _dynamics.2.linear.bias
[Base] _dynamics.2.ln.weight
[Base] _dynamics.2.ln.bias
[Base] _dynamics.4.weight
[Base] _dynamics.4.bias
[LoRA] _reward.0.lora_A
[LoRA] _reward.0.lora_B
[Base] _reward.0.linear.weight
[Base] _reward.0.linear.bias
[Base] _reward.0.ln.weight
[Base] _reward.0.ln.bias
[LoRA] _reward.2.lora_A
[LoRA] _reward.2.lora_B
[Base] _reward.2.linear.weight
[Base] _reward.2.linear